# Class 08 — Multiple Testing
**DSC 215 · Statistical Critical Thinking & Experimental Design · UCSD**

---

When you run a single hypothesis test at $\alpha = 0.05$, you accept a 5% chance of a false positive. Run 10,000 tests simultaneously and you expect 500 false positives by pure chance — even if absolutely nothing is real. This notebook builds the full picture: why it happens, how to measure it, how to correct for it, and what the p-value distribution tells you.

**Contents:**
1. [Setup](#1)
2. [Expected False Positives Under the Complete Null](#2)
3. [FWER — Exact Formula and Approximation](#3)
4. [Bonferroni Correction — Implementation and Power Cost](#4)
5. [P-value Distribution Under H₀ — Proof by Simulation](#5)
6. [P-value Distribution Under Hₐ](#6)
7. [P-value Histogram Diagnostics](#7)
8. [Benford's Law](#8)

---

<a id='1'></a>
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from statsmodels.stats.multitest import multipletests

np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
BLUE   = '#2563EB'
RED    = '#DC2626'
GREEN  = '#16A34A'
ORANGE = '#D97706'
GRAY   = '#6B7280'
PURPLE = '#7C3AED'

print('Ready.')

<a id='2'></a>
## 2. Expected False Positives Under the Complete Null

Under the **complete null** (every $H_{0k}$ is true), each test independently has probability $\alpha$ of a false rejection. By linearity of expectation:

$$E[\text{# false rejections}] = m\alpha$$

This scales linearly with $m$ — a genomics study with 10,000 SNP tests at $\alpha = 0.05$ expects 500 false positives.

In [ ]:
alpha = 0.05
m_values = [1, 10, 100, 1000, 10000, 100000]

print(f'Expected false positives under complete null (α = {alpha})')
print(f'{"m":>10}  {"E[FP] = mα":>15}  {"FWER = 1-(1-α)^m":>20}')
print('-' * 50)
for m in m_values:
    efp  = m * alpha
    fwer = 1 - (1 - alpha)**m
    print(f'{m:>10,}  {efp:>15,.1f}  {fwer:>20.4f}')

In [ ]:
# Simulate to verify: run m=1000 tests all under H0, count false positives
m_sim      = 1000
n_sim      = 5000   # number of simulated experiments
alpha_sim  = 0.05

# Under H0: Z ~ N(0,1), so p-values ~ Uniform(0,1)
p_matrix  = np.random.uniform(0, 1, size=(n_sim, m_sim))
fp_counts = (p_matrix < alpha_sim).sum(axis=1)

print(f'Simulation: {n_sim:,} experiments, m={m_sim} tests each, α={alpha_sim}')
print(f'  Theoretical E[FP] = {m_sim * alpha_sim:.1f}')
print(f'  Simulated  E[FP] = {fp_counts.mean():.2f}  (std={fp_counts.std():.2f})')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(fp_counts, bins=40, color=BLUE, alpha=0.8, edgecolor='white')
ax.axvline(m_sim * alpha_sim, color=RED, lw=2.5, ls='--',
           label=f'Theoretical mean = {m_sim*alpha_sim:.0f}')
ax.axvline(fp_counts.mean(), color=ORANGE, lw=2, ls=':',
           label=f'Simulated mean = {fp_counts.mean():.1f}')
ax.set_xlabel('Number of false positives')
ax.set_ylabel('Frequency across experiments')
ax.set_title(f'Distribution of False Positives Under Complete Null\n'
             f'(m={m_sim}, α={alpha_sim}, {n_sim:,} simulated experiments)')
ax.legend()
plt.tight_layout()
plt.savefig('figures/01_expected_fp.png', bbox_inches='tight')
plt.show()
print('✓ Saved: figures/01_expected_fp.png')

<a id='3'></a>
## 3. FWER — Exact Formula and Approximation

The **Family-Wise Error Rate** is the probability of *any* false positive:

$$\text{FWER} = P_{H_0}[\text{at least one false rejection}] = 1 - (1-\alpha)^m$$

For small $\alpha$, a Taylor expansion gives $\text{FWER} \approx m\alpha$. Both converge — the exact formula saturates at 1, while $m\alpha$ is the linear approximation valid for small $m\alpha$.

In [ ]:
m_range = np.arange(0, 101)
alpha   = 0.05

fwer_exact = 1 - (1 - alpha)**m_range
fwer_approx = m_range * alpha   # linear approx
fwer_approx_clipped = np.minimum(fwer_approx, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(m_range, fwer_exact,         color=BLUE,   lw=2.5, label='Exact: $1-(1-α)^m$')
ax.plot(m_range, fwer_approx_clipped, color=RED, lw=2, ls='--', label='Approx: $mα$ (clipped at 1)')
ax.axhline(0.05, color=GRAY, ls=':', lw=1.5, label='α = 0.05')
ax.axhline(0.50, color=GRAY, ls=':', lw=1.5, label='FWER = 0.50')

# Annotate m where FWER first crosses 50%
m_50 = int(np.argmax(fwer_exact >= 0.5))
ax.axvline(m_50, color=ORANGE, lw=1.5, ls='--')
ax.text(m_50 + 1, 0.35, f'm={m_50}\nFWER≥50%', color=ORANGE, fontsize=9)

ax.set_xlabel('Number of hypotheses tested, m')
ax.set_ylabel('FWER')
ax.set_title('FWER Grows Rapidly with m\n(α = 0.05 per test)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(fontsize=9)

# Right: compare FWER across alpha levels
ax2 = axes[1]
for a, col in [(0.05, RED), (0.01, ORANGE), (0.001, BLUE), (0.0001, GREEN)]:
    ax2.plot(m_range, 1-(1-a)**m_range, color=col, lw=2,
             label=f'α = {a}')
ax2.axhline(0.05, color=GRAY, ls='--', lw=1)
ax2.set_xlabel('Number of hypotheses m')
ax2.set_ylabel('FWER')
ax2.set_title('FWER by Per-Test α Level')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax2.legend(fontsize=9)

plt.suptitle('Family-Wise Error Rate: P(any false positive)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/02_fwer.png', bbox_inches='tight')
plt.show()
print('✓ Saved: figures/02_fwer.png')

<a id='4'></a>
## 4. Bonferroni Correction — Implementation and Power Cost

**Bonferroni** controls FWER at $\alpha^*$ by testing each hypothesis at $\alpha = \alpha^*/m$.

It uses the **union bound** (Boole's inequality), which holds for *any* dependence structure:

$$\text{FWER} = P\left[\bigcup_{k=1}^m \text{reject } H_{0k}\right] \leq \sum_{k=1}^m P[\text{reject } H_{0k}] = m\alpha$$

**Trade-off:** It's guaranteed to control FWER, but it's conservative — it loses power, especially when tests are positively correlated (the true FWER is then below $\alpha^*$ but Bonferroni doesn't know this).

In [ ]:
def run_multiple_tests(m=1000, n_true=50, effect_size=2.0,
                       n_per_group=30, alpha_star=0.05, seed=42):
    """
    Simulate m two-sample t-tests.
    n_true hypotheses have a real effect (effect_size).
    Returns p-values and ground truth labels.
    """
    rng = np.random.default_rng(seed)
    is_true = np.zeros(m, dtype=bool)
    is_true[:n_true] = True
    rng.shuffle(is_true)

    p_values = np.zeros(m)
    for k in range(m):
        mu = effect_size if is_true[k] else 0.0
        group_a = rng.normal(0,   1, n_per_group)
        group_b = rng.normal(mu,  1, n_per_group)
        _, p_values[k] = stats.ttest_ind(group_a, group_b)

    return p_values, is_true

m, n_true = 1000, 50
p_vals, is_true = run_multiple_tests(m=m, n_true=n_true, effect_size=2.0)

alpha_star   = 0.05
alpha_bonf   = alpha_star / m

reject_naive = p_vals < alpha_star
reject_bonf  = p_vals < alpha_bonf

def summarise(reject, is_true, label):
    tp = (reject &  is_true).sum()
    fp = (reject & ~is_true).sum()
    fn = (~reject &  is_true).sum()
    power = tp / is_true.sum()
    fdr   = fp / max(reject.sum(), 1)
    fwer_sim = fp > 0
    print(f'{label}')
    print(f'  Threshold : {alpha_star if "naive" in label.lower() else alpha_bonf:.5f}')
    print(f'  Rejections: {reject.sum()}  (TP={tp}, FP={fp}, FN={fn})')
    print(f'  Power     : {power:.1%}')
    print(f'  FDR       : {fdr:.1%}')
    print(f'  Any FP?   : {"YES" if fwer_sim else "NO"}')
    print()

summarise(reject_naive, is_true, 'Naive (no correction)')
summarise(reject_bonf,  is_true, 'Bonferroni correction')

In [ ]:
# Power vs. m trade-off: how much does Bonferroni cost?
effect_sizes = [1.0, 2.0, 3.0]
m_range_power = np.array([10, 50, 100, 500, 1000, 5000, 10000])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
for es, col in zip(effect_sizes, [RED, BLUE, GREEN]):
    # Theoretical power of a one-sample z-test after Bonferroni
    powers = []
    for m_val in m_range_power:
        alpha_adj = 0.05 / m_val
        z_crit    = stats.norm.ppf(1 - alpha_adj)
        power     = 1 - stats.norm.cdf(z_crit - es * np.sqrt(30))
        powers.append(power)
    ax.semilogx(m_range_power, powers, color=col, lw=2, marker='o', ms=5,
                label=f'Effect size θ = {es}')

ax.axhline(0.8, color=GRAY, ls='--', lw=1, label='80% power target')
ax.set_xlabel('Number of tests m (log scale)')
ax.set_ylabel('Power after Bonferroni')
ax.set_title('Bonferroni Power Cost\n(n=30 per group, α*=0.05)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(fontsize=9)

# Right: p-value scatter showing where rejections fall
ax2 = axes[1]
idx_sorted = np.argsort(p_vals)
p_sorted   = p_vals[idx_sorted]
true_sorted = is_true[idx_sorted]

colors_pts = [GREEN if t else GRAY for t in true_sorted]
ax2.scatter(range(len(p_sorted)), p_sorted, c=colors_pts, s=8, alpha=0.7)
ax2.axhline(alpha_star, color=RED,  lw=2, ls='--', label=f'Naive α = {alpha_star}')
ax2.axhline(alpha_bonf, color=BLUE, lw=2, ls='--', label=f'Bonferroni α = {alpha_bonf:.4f}')
ax2.set_yscale('log')
ax2.set_xlabel('Hypothesis rank (sorted by p-value)')
ax2.set_ylabel('P-value (log scale)')
ax2.set_title('Sorted P-values: True Effects (green) vs. Nulls (gray)\n'
              f'm={m}, {n_true} true effects, effect size=2.0')
ax2.legend(fontsize=9)

plt.suptitle('Bonferroni Correction: FWER Control at the Cost of Power',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/03_bonferroni.png', bbox_inches='tight')
plt.show()
print('✓ Saved: figures/03_bonferroni.png')

<a id='5'></a>
## 5. P-value Distribution Under H₀ — Proof by Simulation

**Theorem:** Under $H_0$, if $F_0$ is continuous, then $P \sim \text{Uniform}(0,1)$.

**Proof sketch:** The p-value is $P = 1 - F_0(T^*)$. Applying $F_0$ (a monotone transform) to both sides:

$$P_{H_0}[P \leq p] = P_{H_0}[T^* \geq F_0^{-1}(1-p)] = 1 - F_0[F_0^{-1}(1-p)] = p$$

This is the CDF of Uniform(0,1). We verify by simulation across three different null distributions.

In [ ]:
N_SIMS = 50000

# Three very different null distributions — p-values should all be Uniform
null_tests = {
    'Z-test\n(Normal null)': lambda: stats.norm.sf(np.random.normal(0,1,N_SIMS)),
    't-test (df=5)\n(Heavy-tailed null)': lambda: stats.t.sf(np.abs(np.random.standard_t(5, N_SIMS)), df=5) * 2,
    'Chi-squared test\n(Skewed null)': lambda: stats.chi2.sf(np.random.chisquare(3, N_SIMS), df=3),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (title, gen) in zip(axes, null_tests.items()):
    pvals = gen()
    ax.hist(pvals, bins=40, density=True, color=BLUE, alpha=0.8, edgecolor='white')
    ax.axhline(1.0, color=RED, lw=2, ls='--', label='Uniform(0,1) density')
    # KS test to verify uniformity
    ks_stat, ks_p = stats.kstest(pvals, 'uniform')
    ax.set_title(f'{title}\nKS test: stat={ks_stat:.3f}, p={ks_p:.3f}')
    ax.set_xlabel('P-value')
    ax.set_ylabel('Density')
    ax.set_ylim(0, 2)
    ax.legend(fontsize=8)

plt.suptitle('P-values Are Uniform(0,1) Under H₀ — Regardless of the Test Statistic',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/04_pvalue_null.png', bbox_inches='tight')
plt.show()
print('✓ Saved: figures/04_pvalue_null.png')

<a id='6'></a>
## 6. P-value Distribution Under Hₐ

For a one-sided Z-test with true effect $\theta$:

$$P_{H_A}[P \leq p] = 1 - \Phi[\Phi^{-1}(1-p) - \theta]$$

This distribution is **stochastically smaller** than Uniform — p-values pile up near 0. The larger $\theta$, the more extreme the leftward skew. This is what gives the test power: under the alternative, small p-values are much more likely.

In [ ]:
# CDF of p-values under HA: P_HA[P <= p] = 1 - Phi(Phi^{-1}(1-p) - theta)
p_grid = np.linspace(0.001, 0.999, 500)
thetas = [0, 0.5, 1.0, 2.0, 3.0]
colors_th = [GRAY, PURPLE, BLUE, ORANGE, RED]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: CDF of p-values (from the slides)
ax = axes[0]
for theta, col in zip(thetas, colors_th):
    cdf = 1 - stats.norm.cdf(stats.norm.ppf(1 - p_grid) - theta)
    label = f'θ={theta} ({"null" if theta==0 else "alt"}.)'
    ax.plot(p_grid, cdf, color=col, lw=2, label=label)
ax.plot([0,1],[0,1], 'k--', lw=1, label='Uniform (θ=0 exact)')
ax.set_xlabel('p')
ax.set_ylabel('$P_{H_A}[P \\leq p]$')
ax.set_title('CDF of P-values Under H₀ and Hₐ\n(Replicating slide figure)')
ax.legend(fontsize=9)

# Right: PDF/histogram view
ax2 = axes[1]
for theta, col in zip([0, 1.0, 2.0, 3.0], [GRAY, BLUE, ORANGE, RED]):
    # Simulate p-values
    z_obs = np.random.normal(theta, 1, 30000)
    pvals_ha = stats.norm.sf(z_obs)  # one-sided
    ax2.hist(pvals_ha, bins=50, density=True, alpha=0.5, color=col,
             label=f'θ = {theta}')
ax2.axhline(1.0, color='black', ls='--', lw=1.5, label='Uniform(0,1)')
ax2.set_xlabel('P-value')
ax2.set_ylabel('Density')
ax2.set_title('P-value Distribution by Effect Size')
ax2.legend(fontsize=9)

plt.suptitle('P-values Pile Up Near 0 Under the Alternative — Larger θ = More Extreme',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/05_pvalue_alternative.png', bbox_inches='tight')
plt.show()
print('✓ Saved: figures/05_pvalue_alternative.png')

<a id='7'></a>
## 7. P-value Histogram Diagnostics

In a multiple testing study, the shape of the p-value histogram is a diagnostic tool. Three canonical patterns:

| Histogram shape | Interpretation |
|-----------------|----------------|
| Flat / uniform | Mostly nulls, no signal |
| Spike near 0 + flat | Mix of true effects and nulls (healthy) |
| Spike near 0 + elevated everywhere | Inflated — calibration problem, anti-conservative test |

In [ ]:
m_diag = 5000

def simulate_pvals(m, frac_true, theta, inflate=False):
    """Simulate m p-values with frac_true true effects of size theta."""
    n_true = int(m * frac_true)
    # Null p-values: uniform
    p_null = np.random.uniform(0, 1, m - n_true)
    # Alt p-values: pile near 0
    z_alt  = np.random.normal(theta, 1, n_true)
    p_alt  = stats.norm.sf(z_alt)
    pvals  = np.concatenate([p_null, p_alt])
    if inflate:
        # Simulate inflation: compress p-values toward 0
        pvals = pvals ** 1.8
    return pvals

scenarios = [
    ('All null\n(no signal)',            simulate_pvals(m_diag, 0.0,  0.0),  GRAY),
    ('5% true effects\n(θ=2, healthy)', simulate_pvals(m_diag, 0.05, 2.0),  BLUE),
    ('5% true effects\n(inflated)',     simulate_pvals(m_diag, 0.05, 2.0, inflate=True), RED),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (title, pv, col) in zip(axes, scenarios):
    ax.hist(pv, bins=40, density=True, color=col, alpha=0.8, edgecolor='white')
    ax.axhline(1.0, color='black', lw=2, ls='--', label='Uniform(0,1)')
    ax.set_xlabel('P-value')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.set_ylim(0, ax.get_ylim()[1])
    ax.legend(fontsize=8)
    # Annotate the spike region
    frac_below = (pv < 0.05).mean()
    ax.text(0.5, ax.get_ylim()[1]*0.85,
            f'{frac_below:.1%} of tests\np < 0.05',
            ha='center', fontsize=9,
            color='black')

plt.suptitle('P-value Histogram Shapes as Diagnostics',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/06_pvalue_diagnostics.png', bbox_inches='tight')
plt.show()
print('✓ Saved: figures/06_pvalue_diagnostics.png')

<a id='8'></a>
## 8. Benford's Law

The slides end with Benford's Law — a natural application of distributional reasoning to real data. In naturally occurring numerical datasets spanning multiple orders of magnitude, the leading digit $d$ follows:

$$P(d) = \log_{10}\left(1 + \frac{1}{d}\right)$$

**Why?** If data spans many orders of magnitude and is roughly log-uniform, the fractional part of the log is uniform — which gives Benford's distribution for the leading digit.

**Use in statistics:** Significant deviation from Benford's law in financial or scientific data is a signal of fabrication or rounding. Used in forensic accounting and as a sanity check on reported experimental results.

In [ ]:
# Benford's law PMF
digits   = np.arange(1, 10)
benford  = np.log10(1 + 1/digits)

print('Benford\'s Law PMF:')
for d, p in zip(digits, benford):
    bar = '█' * int(p * 100)
    print(f'  d={d}: {p:.3f}  {bar}')

# Verify on naturally occurring data: Fibonacci numbers
def fibonacci(n):
    a, b = 1, 1
    fibs = [a]
    for _ in range(n-1):
        a, b = b, a + b
        fibs.append(a)
    return fibs

fibs = fibonacci(5000)
leading_fibs = [int(str(f)[0]) for f in fibs]
fib_counts   = np.array([leading_fibs.count(d) for d in digits]) / len(fibs)

# Simulate "fabricated" data — uniform leading digits (humans tend to do this)
n_fab = 5000
fake_leading = np.random.choice(digits, size=n_fab)
fake_counts  = np.array([(fake_leading == d).sum() for d in digits]) / n_fab

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
width = 0.3
ax.bar(digits - width/2, benford,    width, color=BLUE,  alpha=0.8, label="Benford's law")
ax.bar(digits + width/2, fib_counts, width, color=GREEN, alpha=0.8, label='Fibonacci numbers')
ax.set_xticks(digits)
ax.set_xlabel('Leading digit')
ax.set_ylabel('Probability')
ax.set_title("Fibonacci Numbers Follow Benford's Law")
ax.legend()

ax2 = axes[1]
ax2.bar(digits - width/2, benford,     width, color=BLUE, alpha=0.8, label="Benford's law")
ax2.bar(digits + width/2, fake_counts, width, color=RED,  alpha=0.8, label='Fabricated (uniform)')
ax2.axhline(1/9, color=RED, ls=':', lw=1.5, label='Uniform = 1/9 each')

# Chi-squared test for fabricated data
expected = benford * n_fab
observed = fake_counts * n_fab
chi2_stat, chi2_p = stats.chisquare(observed, f_exp=expected)
ax2.set_title(f"Fabricated Data Deviates from Benford's Law\n"
              f'χ² test: stat={chi2_stat:.1f}, p={chi2_p:.2e}')
ax2.set_xticks(digits)
ax2.set_xlabel('Leading digit')
ax2.set_ylabel('Probability')
ax2.legend(fontsize=9)

plt.suptitle("Benford's Law: A Distributional Test for Data Authenticity",
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/07_benford.png', bbox_inches='tight')
plt.show()
print('✓ Saved: figures/07_benford.png')

---

## 📝 Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **E[FP] = mα** | Under complete null, false positives grow linearly with number of tests |
| **FWER = 1-(1-α)^m** | Probability of *any* false positive — saturates at 1 quickly |
| **Bonferroni** | Use α/m per test — controls FWER under any dependence, but conservative |
| **P-values ~ Uniform(0,1) under H₀** | This is a theorem, not an assumption — follows from probability integral transform |
| **P-values pile near 0 under Hₐ** | CDF is $1 - \Phi[\Phi^{-1}(1-p) - \theta]$ — leftward skewed |
| **P-value histograms are diagnostic** | Flat=all null, spike+flat=signal present, spike+elevated=inflation |
| **Benford's law** | Leading digit distribution for natural data — deviations signal fabrication |

---

## 📚 References

- Schwartzman, A. DSC 215 Lecture Notes, UCSD Spring 2026.
- Benjamini, Y. & Hochberg, Y. (1995). *Controlling the false discovery rate: a practical and powerful approach to multiple testing.* JRSS-B 57(1): 289–300.
- Bland, J.M. & Altman, D.G. (1995). *Multiple significance tests: the Bonferroni method.* BMJ 310: 170.
- Benford, F. (1938). *The law of anomalous numbers.* Proc. American Philosophical Society 78(4): 551–572.

---
*DSC 215 · UCSD · Spring 2026 · Prof. Armin Schwartzman*